In [1]:
%%capture
!pip -q install pandas kagglehub

In [2]:
import kagglehub
import os
import glob
import pandas as pd

ruta = kagglehub.dataset_download(
    "adktyakirloskar/movies"
)

print("Dataset descargado en:", ruta)

archivos_csv = glob.glob(
    os.path.join(ruta, "**", "*.csv"),
    recursive=True
)

print("Archivos encontrados:", archivos_csv)

100%|██████████| 1.58M/1.58M [00:00<00:00, 110MB/s]

Extracting files...
Dataset descargado en: /root/.cache/kagglehub/datasets/adktyakirloskar/movies/versions/1
Archivos encontrados: ['/root/.cache/kagglehub/datasets/adktyakirloskar/movies/versions/1/tmdb_5000_movies.csv']


In [3]:
archivo = next(
    (f for f in archivos_csv
     if os.path.basename(f) == "movies_database.csv"),
    archivos_csv[0] if archivos_csv else None
)

if archivo is None:
    raise FileNotFoundError(
        "No se encontró un CSV. "
        "Descárgalo desde Kaggle y súbelo a Colab."
    )

df = pd.read_csv(archivo)

print("Dimensiones:", df.shape)
display(df.head())
print("\nColumnas:")
print(df.columns.tolist())
df.info()

Dimensiones: (4803, 20)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124



Columnas:
['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-nul

In [ ]:
df.columns = df.columns.str.strip()

df = df.drop_duplicates().copy()

if "ts" in df.columns:
    df["ts"] = pd.to_datetime(
        df["ts"], errors="coerce", utc=True
    )

df = df.dropna(subset=["device"]).copy()

# Convertir los valores NumPy a tipos Python
# compatibles con la serialización BSON.
def preparar_documento(registro):
    documento = registro.dropna().to_dict()

    for clave, valor in documento.items():
        if hasattr(valor, "item"):
            documento[clave] = valor.item()
        elif isinstance(valor, pd.Timestamp):
            documento[clave] = valor.to_pydatetime()

    return documento

documentos = [
    preparar_documento(fila)
    for _, fila in df.head(5000).iterrows()
]

print("Documentos preparados:", len(documentos))
print(documentos[0])

/tmp/ipykernel_6868/4172799871.py:21: UserWarning: Discarding nonzero nanoseconds in conversion.
  documento[clave] = valor.to_pydatetime()


Documentos preparados: 5000
{'ts': datetime.datetime(1970, 1, 1, 0, 0, 1, 594512, tzinfo=datetime.timezone.utc), 'device': 'b8:27:eb:bf:9d:51', 'co': 0.0049559386483912, 'humidity': 51.0, 'light': False, 'lpg': 0.0076508222705571, 'motion': False, 'smoke': 0.0204112701224129, 'temp': 22.7}


In [ ]:
df["ts"] = pd.to_datetime(
    df["ts"],
    unit="s",
    errors="coerce",
    utc=True
)

print(df["ts"].head())
print("Valores faltantes:\n", df.isna().sum())
print("Duplicados:", df.duplicated().sum())
print("Dispositivos:", df["device"].nunique())

0   1970-01-01 00:00:01.594512094+00:00
1   1970-01-01 00:00:01.594512094+00:00
2   1970-01-01 00:00:01.594512098+00:00
3   1970-01-01 00:00:01.594512099+00:00
4   1970-01-01 00:00:01.594512101+00:00
Name: ts, dtype: datetime64[ns, UTC]
Valores faltantes:
 ts          0
device      0
co          0
humidity    0
light       0
lpg         0
motion      0
smoke       0
temp        0
dtype: int64
Duplicados: 0
Dispositivos: 3


In [ ]:
documento = df.iloc[0].to_dict()

print(documento)
print(type(documento))

{'ts': Timestamp('1970-01-01 00:00:01.594512094+0000', tz='UTC'), 'device': 'b8:27:eb:bf:9d:51', 'co': 0.0049559386483912, 'humidity': 51.0, 'light': False, 'lpg': 0.0076508222705571, 'motion': False, 'smoke': 0.0204112701224129, 'temp': 22.7}
<class 'dict'>


In [ ]:
{
    "device": "sensor_01",
    "ts": "2020-07-12T00:00:00Z",
    "temp": 22.7,
    "humidity": 51.0,
    "motion": False
}

{'device': 'sensor_01',
 'ts': '2020-07-12T00:00:00Z',
 'temp': 22.7,
 'humidity': 51.0,
 'motion': False}